# LexIA — Jurisprudência — BGE-M3 na GPU do Kaggle

Notebook privado para configurar a GPU, comprovar compatibilidade com legislação e gerar embeddings de lotes validados. Nesta primeira execução, rode até o benchmark: nenhum documento de jurisprudência é necessário para configurar a GPU.

**Configuração no Kaggle:** GPU disponível em Settings → Accelerator, Internet habilitada para baixar o snapshot oficial do modelo e notebook privado. Se Accelerator estiver desabilitado, conclua pessoalmente a verificação da conta. Não coloque senha, token de API ou conexão do PostgreSQL neste notebook.

A produção exige o notebook 05 concluído e pacotes privados exportados dessa versão. Os resultados serão baixados para validação e indexação local no pgvector; não se expõe o banco do computador à Internet.

In [ ]:
import os,sys,json,hashlib,time,base64,subprocess,importlib.util
from pathlib import Path
import numpy as np
import torch
assert torch.cuda.is_available(), 'GPU não habilitada: verifique a conta e Settings → Accelerator.'
DEVICE='cuda:0'
print('PyTorch:',torch.__version__)
print('GPUs disponíveis:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()):print(i,torch.cuda.get_device_name(i))
print('Esta execução usa somente',DEVICE,'; T4 x2 não significa uso automático das duas GPUs.')
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False

## Dependências

Preserve o PyTorch com CUDA fornecido pelo Kaggle. A célula instala somente dependências ausentes pelo pip; não instala uma versão CPU do PyTorch. As versões efetivas são registradas para reproduzir a execução.

In [ ]:
required={'sentence_transformers':'sentence-transformers','huggingface_hub':'huggingface-hub','pyarrow':'pyarrow'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from importlib.metadata import version
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
import pyarrow as pa
import pyarrow.parquet as pq
VERSIONS={name:version(name) for name in ['torch','numpy','transformers','sentence-transformers','huggingface-hub','pyarrow']}
print(VERSIONS)
assert torch.cuda.is_available(), 'O ambiente deixou de disponibilizar CUDA.'

## Snapshot fixado e precisão

São utilizados os arquivos exatos do backbone já adotado em legislação: BGE-M3, saída densa, 1.024 dimensões e normalização L2. O modelo é baixado para `/tmp`, fora da pasta de resultados, e cada arquivo é confrontado com seu SHA-256 original. Os pesos permanecem em float32 e TF32 fica desabilitado.

**Precisão de cálculo:** por decisão registrada após o benchmark na T4, a produção usa **FP16 (autocast)**, cerca de 4,5 vezes mais rápido que float32, com cosseno mínimo de 0,999999 no teste. Os vetores são renormalizados e **gravados em float32**. A compatibilidade com a legislação é exigida também em FP16, e cada parte recalcula uma amostra em float32 como conferência (cosseno mínimo de 0,9995; ajustado de 0,9999 após uma parte real atingir 0,999865).

In [ ]:
MODEL='BAAI/bge-m3'
REVISION='5617a9f61b028005a4858fdac845db406aefb181'
EXPECTED_FILES={'config.json': '26159e7ad065073448460117eb24b7a4572f6f4e78eadff65dc0a11c052449fa', 'modules.json': '84e40c8e006c9b1d6c122e02cba9b02458120b5fb0c87b746c41e0207cf642cf', 'config_sentence_transformers.json': '1eef72430e7194a1e59680e635aed81ffa083f05668dbc5bb1c56c04c0999c38', 'sentence_bert_config.json': 'eb9b44b13c0f52a3b3685c3b1cbdea1ba8b04bea123b98f61610048940776eb1', 'tokenizer.json': '21106b6d7dab2952c1d496fb21d5dc9db75c28ed361a05f5020bbba27810dd08', 'tokenizer_config.json': 'a62b2b6784f990259fddef5f16388693a8043be4f69179e6a5257eeb3f9abac4', 'special_tokens_map.json': '8c785abebea9ae3257b61681b4e6fd8365ceafde980c21970d001e834cf10835', 'sentencepiece.bpe.model': 'cfc8146abe2a0488e9e2a0c56de7952f7c11ab059eca145a0a727afce0db2865', '1_Pooling/config.json': 'e54c164a07274f2eb45bb724f54a79d1efcc90c41573887cd9a29aeee0597352', 'pytorch_model.bin': 'b5e0ce3470abf5ef3831aa1bd5553b486803e83251590ab7ff35a117cf6aad38'}
PROBES=[{'texto': '[codigos/codigo-civil-lei-10406-2002.md | PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO III — Do Domicílio > Art. 70]\n**Art. 70. O** domicílio da pessoa natural é o lugar onde ela estabelece a sua residência com ânimo definitivo.\n\n', 'tokens': [378, 587, 4803, 232, 64, 587, 4803, 31, 9, 318, 5115, 9, 12365, 9193, 150744, 9, 41632, 5, 54686, 6, 58745, 436, 102826, 119899, 7842, 977, 6, 130296, 14255, 87, 292, 88693, 115266, 13055, 9713, 977, 384, 104575, 142666, 4830, 292, 984, 9343, 86597, 10258, 977, 4713, 5, 2358, 268, 16459, 23825, 5, 2358, 5, 180, 25442, 4539, 86597, 10258, 48, 29956, 6083, 393, 36, 4487, 5321, 10301, 145575, 10, 1646, 193536, 375, 3797, 18092, 44836, 31, 5], 'sha256': 'ba6926d5138bfb14ecebd0cdc09e00336ee314ec3287474b917f8478a1c2d465'}, {'texto': '[codigos/codigo-civil-lei-10406-2002.md | PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO III — Do Domicílio > Art. 74]\n**Art. 74.** Muda-se o domicílio, transferindo a residência, com a intenção manifesta de o mudar.\n\n', 'tokens': [378, 587, 4803, 232, 64, 587, 4803, 31, 9, 318, 5115, 9, 12365, 9193, 150744, 9, 41632, 5, 54686, 6, 58745, 436, 102826, 119899, 7842, 977, 6, 130296, 14255, 87, 292, 88693, 115266, 13055, 9713, 977, 384, 104575, 142666, 4830, 292, 984, 9343, 86597, 10258, 977, 4713, 5, 15124, 268, 16459, 23825, 5, 15124, 5, 25442, 56049, 9, 184, 36, 4539, 86597, 10258, 4, 12302, 36735, 10, 193536, 4, 375, 10, 226411, 20243, 8, 36, 82296, 5], 'sha256': '900cac88f7d6a41c7d4be2004e7be9f597d8c8f2cb14f0132017170918d7591d'}, {'texto': '[codigos/codigo-civil-lei-10406-2002.md | PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO III — Do Domicílio > Art. 76]\n**Art. 76.** Têm domicílio necessário o incapaz, o servidor público, o militar, o marítimo e o preso.\n\n', 'tokens': [378, 587, 4803, 232, 64, 587, 4803, 31, 9, 318, 5115, 9, 12365, 9193, 150744, 9, 41632, 5, 54686, 6, 58745, 436, 102826, 119899, 7842, 977, 6, 130296, 14255, 87, 292, 88693, 115266, 13055, 9713, 977, 384, 104575, 142666, 4830, 292, 984, 9343, 86597, 10258, 977, 4713, 5, 14810, 268, 16459, 23825, 5, 14810, 5, 25442, 97408, 39, 4539, 86597, 10258, 46928, 36, 18152, 42796, 4, 36, 113468, 10304, 4, 36, 33796, 4, 36, 226725, 28, 36, 29377, 5], 'sha256': '4c0f191af9be0103c3d2cea563fe601484ea113837dff913301ae91d994b31de'}]
PROBE_VECTORS_B64='/JWDvGXtzDvxb3u8rQv6vKEvL71nALa9V7axPFMqvDutbnI8K5BjPC+E4DsM/w48d/9DvcEXhbyDwEu9StyCPFjgdDxc+5y8ZjrsvA7XrrswDYC80RgxvdPqAT0uw0M8yg0cvcH8+Tl3jiw96OnhPHb5q7zxCeU6Tz+rvKT427xwkhs9+MQJPF2GqLsT3608QyKFvGyi+7vLcw+9srJbO1YMFD1Htxw9oPyZu4Kop7oQxJo7myEXveu7mbuMtEK9bbrju5gpGb0VIEa8XsiDO0MXWT1bTu68gEx0vXrnxD18h1W9ACuAvSXi8jxyuHu9pkY/vAwxvzyvsIG9dESivA3SXbwYlKC8/+JWt4VkB7zYgsq86REovThUljxy9588+kFwvbqprbz7vBa95vXpPDcrGjzP7Di9yo2bvf3nC7wBU8u5qVJFPBQ5uTxtmVG8u6IdvDYYLjzgKIy82KyDPfuVyDwJcT69kIhuvfEbnLysYl67Sv/Wu8y9p7wg3oe8TXkjPbVkDb2q72k8qJIZPYnqqbskr4o7em1gPAmanLyFtp28us8HvZBDpjyMU6w7dHYjPRDxuzscNNI7HbKNu+RKBz2cDii8YF1bu0hHR7pGE4e7dVwRvcJ14LxJGGa7KnKBPG/2UD3YbK07cpeOvANutLwmcTg9ZD+ZPPDLCzyMBvy7bWH3PPEbvDzLLzs9fvEIvU3izrwgSgO9XwlDu+cCvTxeomU9q+eGPLk+AbvKKIo8JkCYPJFgc7xnhWO9ynZIPDrAF7piIh48AZ07vZMlzTyC9wG7m0bfPPdvej2wSC47bFvePAAaUrsvJow8uXlivF/ycrv1YjQ9//fyvJG6SztulYq8GKwFPfn70ryl7ya8ErMqvbDssDzn02k95saSOwD0C7zBsy499THnvHSt+zqkZw08VLeHPO1fJj1Q1uw8E9Y5PE/20LujYHu8lM1OvNVRND19kG+9/25rPSapIb2j3C895rzLvPN+4jrbLa88mFguPTZJsjsFLGq85krXvNy8BjyXtLs8B6a4vDCjkLxdSyQ7IakvvEELyLvmzF69ShmsPAvaJD1rIaa8Wo+GO0ccEbyxzZO9U+8QPNM/Ij0Qeru9l0iYvKFr6DsX03U84KHWu32lJr1oiu87W6w1vEZPeL048209pl+CPEXXEDwWuQK8Ye6tPH0VkjxG6Lg8JrcMvafGh7zPtQO9CoZePTfGNTw4UhW99qeuPNejdTsta+680lM2PU6RzLyvoZK9nlYqPUTMIzvrDpG8DzNcvB7xsLzU3SQ96vQsvF9UXDwjtPi7GKYCvXDBwbl2HpG8HmR8vNsvqLzzaAq94YoqPR+ORTz8mro8XsEGvM6T+LxkrNg6vm43u22dKzuU97y7oeEovPzAkLsPIxs949kCPRGaKb290Qq8k1uHPN2w1jwc89U7JVgPvE8AGr2Sq9g8b5sCvQ0fJjwhkbu7SbadPJFLGDuBY2I9giWJvIaLwTnnNly7MwcIPf2cxrxLwHY9L39tvR+iqbvjRFU8LcodPQAoJ7x5Xs080gn4vLFiLz3MxY67PaSLvd+1vbwcXBi97ywlvj3vDT0Jsnk8ffOEvIl/0zr06VM8r4zPvMjvIL1FbWi72Ud7PTPp7Tz0XxG9zm9pvDbDxzyBhTA8nAPHvNlMybzmFAE8SPmGu99IJTzQ1Ia87ZZkvXZ+ejyxYnu7WUbBPLXYRT3GeWU9hG2DvFfb57zIsNs6SRcQPOGYJ707TiO7NcTRvMO/yTxl6gc96h6LPC5rPr3UucI5xPxGPRyIjrx+4g49TAf/vGbJnT0fcLg7tm8EO2+avbxJabS8S9onvFjJJj2zwRC87N5tPKtV+7swB8y89NpzvUReqDwgonU9WNnzOz7V0jslaOI8lxEKvelN/rzhjoA80coIvXobOb3j5TE9O+IqPUUH6Ty8gQO7CE4nvQwVuLygfOK8z6jCPMzEFT3Jt9O8ZC7AO3JcwTynfyO7JljPu2RQgr3Ojvi8YxibOyNAmLzyXq27Q/ZEvdNfeL1Ufu88oOoEvZY5ijsTAII+JNaTPdab1jueoqO9gOSgugt9Rr21T9q75YAEu8qXELwqeR+9I1pNu+d82TxzAHA8nbNpPGxazrssPkU94mCovF43rrrSlxc9g/mNPL9KGbyViIy8avWLPG3/4jtKW/W8g3v2vNQtVTz5iBC9jvaUvWZ1ybw2Xyw8/BgOvFb+YD2+RoG8omnsPF3l3zmi3Ys8KwZhunrhTL0lheU8rIx6u5lBwLsOxjW9NyP6vIf2KrzkbYo8s7IQPYY+FD16+AA8Y6sEPaHvDD21j6a8WNLFu/QDibxrXEG90JILvDBmqbzIsNU7Opj7u13QZjzcz008JNVhPIPsCj3Txz298w6mPco1Tzxry/08jD4XvZKbED2lvNE8SWjJPEH4Cr1jKMM7E8wNPSktHLs8gIe6z5gyPCZ7+zsgzpg8jjn4Oqv5gzyX3vK8rWxEPQIuHz1Wuio99x9cO0goBDzMMCE9htEFPWlwCzv9juW7+v5PO1pedjyicS89ElEIPPySjryeXxC8woqEPJFTjb0jGaA7zzrqvDLDJLx86TC9xVoYPLhZJj3JqKi9tJ+2ui2/UT1y6Oe84LbFPLoAt7yOUQA7dYgXvHSCT71njHk8UtgpPWui2TzFKaw82usAPe/ofTz27eE8D9ZxvJ/1mTpPNIw8/KXUvP+Kjrty/Gk81bt5PDI/C7v4Wpu8AFmPPAf3GL30RpC9Z5EZPbK/ADw6d+67SXxDPQn7J7zWM4C9/tC2vB/w/bxOZ1k9csmjvPxnkjyBE8+8Q9/NusRFAz2IG1O8v4ZRvMcCjbyVAFu8XagMPdOVdzwhagK9/+YhPf8Bb72c00q80r+Eu7dGRj14SrO8ZYEiPOrjkLuXWQM9+i+TPUSyajxJ+528EosavWCmsDwYaBe82Q2VuqhHzzqt2Ga9OzqCvLHecL2d23O8uY8FPRvDlLwNeZk8AVQFvXefrrxF0cQ7uSNUuzcHuzw+Jhk94WgKPXqbTLxP2pY9rTQuvKA3wT1UdTK60ad5vf0niD3GPRa8KYkKveXAmDvhoZ88WPOPPCe/gj2L0wc9Nz6pvGzgHrwYJYq86R5cuH9hdz0UK6+84pTjvHMmVL2A2+y77hxUvG3IhT0+JY278d6lPDXPDrxirBk9VIGyvBdE8ruiBSy8U9sQPUrM0TxBPFi8wWNRvH91JTyXfVk73zCBvcINeL2glNo8ADoqPYkXATy9VmY7tZXquyT5aLt9+DO9WOqEPMqtGbxOHia8JmVevBj3FjxN24i7gXD3O4MUo7rb2ws9qNh1O1LsMD3uljo9dvCrPG71mT2Cm0+9NpIYvcibATzm48c7XSI3vcMghLx4Uoa7ETqhvG4YVTvQZhc9ivvQOqrmzDtxWl+89stYu3w3QTz2HAw9EPwgPbvXVrtrT4M8XuYtvPPwaz3IfS27vfLCvM/7l737Apa8J10kPFCqOrzQnAe8o+Hdu2aarburkp+8FEKKPAxwhzz8oQa97U0bvMXgk7tTTPM74SSnPCUi4TupFw28b3cfPbmsorvufUE9h2hIu/b16Lwbc+682cIWvTSYB716+dY82WNjPAeSt7xPclY8tMVAvLbzxDjQleG7jNEkPadD1Lx+b0G9Xat7vPb38rrd6TA88ZErvR6wJTxYC+o6a81jPJFTm73sQrI4rz1dPTgUZDz0zTm7qb4GPBeeOzxtDyM8yaWuPCAv1Lvv4tc8YGgqvZ/RBLyokgS91UwyPfLSQLzfW6K7i3w7veS4Ez0FI728gok2vN696bsjTbu774UzvWY4JLwVne07nwkhvW9jqzzPLHg8WFkaPCrewLuOkqU8UY3hPCtZIz1TyZE8SDNpPPHg7LgrjZ69GGxivKZmprxcwr08PPC5OotX2bu5TKo8/u5Vu7acNz2ZqMq8NyHIPI0DZrsBB0y8ifYOPf0iZDsfOCs8W1KHu1e4UrvgJQm85EcKvc14kDxPF7+8gdMYPCw5sTwwbOk8rcwoO1HVSr3yk3u8BLS+vGZEB71diRA8KDhUvFgBNDxeLTm9MAtbvTxwEbwg5kM7GcuUOyW9gjzuhfY8DQQDPafJuDznStW813quvGpqnjyZqF69VK32vVcjQjt3TTq9XdibupnJGL1S+aI8mjJavb1vYLxupxq9beifOn/bL71sA/26HsT+O4twDr3Q1AU9His/PefAPr3Ow3A7zY1zvTAFBj0g73W8SYqfu2hKuzs6cmc7q/ImPdIl97x/xWS9Wly6vJ4sqLxnviS9WAQVPFltEL1ga7q7gY9HOqQUjbwZpt881aMpPWQeP725RMi820r4PGyBYTs/rKc8x/5bOhGiLryJdRS8XMB9PS2TcbvBLTu9Zx2pPH63n7wjB5w8RfViPaXHmbzCWo492puhvEhyMD0Zg9S8fFfTvOu9d73rznU8vYlpvOJtuzz7pyq8dlMwPFCHVDyUTjE9aK5CuxtvuzwR3eq82VbivLdvv7z3T4O8Id0NvNKQvDpwOVq8yXgXvFpUB72G1xO8hXcpvYKmODxBGTe9SrHAPKvPCb3y+RS8NrqPPCsL0TpcsqS8+fHvu8dn+bv/yro6bsYyvRSXCL2jzC69AzoiPFuBErq9zIM8qSMWvUx/+rsSP0Q9sfkUva5QqjzZ6BC9qOCGPD6StDy7piS9fOk2Pb+KhDxEoXE8fADgvEbEMr2lO9s7vGeMPJBZ27wowFG9/S3yO4tKMTzn+Su9lfYCPRCziz0/xCw97wKpup6za7xwGRm7zBLZvKgkuTuM4/O8UQYOvCsvMD1wgog9HrH+vHBYWzz9zz29MycePd2PJD1jjqM8rdWUvAhB/TyjHQ28ei/wvMu5HryoSha8HPpMvXZeijrKR667/5E9PI7dRz3QUsE6FojtPMBHtDnp/1U8QTu0PDdzIr16NB48Hs9TPew7irxFXuU8TavGvHx0hzwCRso8TSNjPOINBjwP4Qe9GfO5uzgYqry2iju9lmUlPcz6ELtK7x+8bMN9vO74ujxVaZO6cI0GPYoUSToaJl28tJkOPQsgmLvv1uK7b0QbPbZgM7weZms784MaPBG6QT0SIv07b5T4ussiDj1xMdA8WJjZPLj9wjwCGG26W+LevHZuN72Jb+i5lbWevGzSZj3I8YQ8wRfPPGaEK7yNHDA9brGlPPtgwboFQOM8FZsSvekOdDy5Rw69r6RzPOkkvbwQLOw8c4zVu/AIAj1cnHw89yvZvPwul7uVjeW7QP+hPRykkboLxRu9T3n6PC1e/DyXz4y7xiWgu2onnDt8Xlg9rVQQvUMF77zXAza9lJI3PAyxY7t5ZRE9CEqGvH5/qDti6xU8UqljvJL0dDwfSMI8xlKCPI4cAzwTuY06D+RAPIHTzTxcxo68TsGjPRmiKL3htnq9HecMvXumPL0THLQ89rfuu2tkBroBEw29ldOFvWqX6jvFXaU6QhT6O3IKsbr2Ux68n81fPBq8Yr0qibK70X0ivfw8mLvGsQs915ucvJDWaDzAJqa7kcKfvHSrJL2Xjqk87kaNOwtwNb1UxMG8idEVPQeZ9DxI9Y08IBo+PBuVDr2Jcfy8llY7PZykkjsI4FW8/QlUPRxj1bvMTuO8eQYJvaOS4DyIShM9u9KMPD2rZbzReIy8E3+OPFUg9bzi98C8qFf9vHoGjrzvGF27mdf5uxqMGLyX5lU9VDuZvdn9Lr1pE4I9tFmMvew3WLy215k8XkeYvV3LVbwRW5+7LtaQvMBAAL2ORjI83ItIPLiA+LilsKM8eE2PvPN9GL0IcbE8CXs7PZJbo7zSgM+8iDxXvYNNTj1FCc68Dc9Ivbn8Zr2xStS72228PCE3Ejxp+pE8MmIsPCdkjryktw08D4eevMT4hj1Qr8c8PDvzvLwyJr1Q2La77SwDvLZxSzzVnHO8YPhEPE7M9TzGr7681CiSudvQlDwy2jc76gruPGxhtzyyHCW9jidTu1owG7xseyQ9Z+lbPSsCjT1yIT48fUSUuhYSxjs/FA09sCSovAFPTrx4oyy8YlqdOfV5KL35xD070Drru7gNeTxwYE094E1RPWV7mbtDS868Gd8CPahUSDy6MQg9voYcva+04jxg0Sw8mhV6PdPNDbstXuC8UDQ3vQC2eDr3nOw8shTaPFcyVzxtehI8kDwEvO+lwLwNasC7+DJQvXOE4zxkGQ69jPkPPY1E7Lq3C7A5iULYPL/WfTw6s2s9tOgFPTPeebwNv6M8EQsFvT4Q9rt5IwS8F/I0PQG3FDwbt7K8lyGqPD1Z+zy+hbC87W4hvCXTiL1B46M8p2KHPXJ45TtR4CG989nMPAVEdjsRtAw9gl5puht+MrojYEk9mMjiPOQi7bpI4Mk4V6k0vUTTFrwF5g89QxmTvYj+aj1gAvS8udoBPTp6Ar2QguY7Atg9PMc1mrstbhM8Ax7kvK9/tLzZpgo9S0vCO8EIL73cMfK8Ulq/OxR/sjzrqaM8pqqavCcTIz3EPdQ8iqILvXebJ7uMeYI6esqPvaT39jttPUY95TqvvY/en7s1zoS8uTWPvI+ay7z5+ty7xDwwOwvL3Lt910e9A4K1PKZdtLzor9g7FiSKPNkgTTzneFm6iPfUusjYuDtKQAs8SMI5POszKT3JlRQ9vsEHvYAXnjp2KkM8qZ0Svd3NWT0v9ky8Cz98vcQgUz18FcG8vJq+ulbaTLy7k6+8dt7aO0/VLrwzwR88GA19PP1DAb1kxeA7L9G5vGFKJ7zhYAG8tbk7vL3HnjzucZs8aha5PE9sADus79y8gwg4ul2F5zuQ4Y+8qEXjPBMpST1vGgA4VhoWPRJ6HD3hUkm8KrrGu6oRXTw3oCs9SesGvEZE2bu6LgW9xLbyPKRY+rwpm847hJQ1u18r2zxDIHC6HNWPPezEYzyq9ZE7B00xusHzKj25NAK9kBlkPYHdir2C3QG90ZI1OR206jy5M8K8SZytPGYsfrzTPE89IEBiOqFchL2e/ae8ZPgGvBZjJr6ukQI9JIAXPIxMlLy9r427YV/aPBocJ70nLMy84A1WvOCfVz095pY8Z0knvZVwB7zpFCo8sh1dvJv7vbxDLwK9jkgrvIq4A7zBN5i8mkKEvN4obL3NhTg8a135OgEXNDxS2yw9E0faPPn3QDwXVtK8PY62PNTiZzxvXti8c76Iu4j6Ybx/qtI7bbA4PdILmDsF4Q48lOqCO/47YD1aoGM7zhBkPazvcrz8rTI9DF3VPANLVrwNiUG9F9K4vJ/0ibw70t48vryavMF9ibrJl9m8PYapvPMWHb1NUNw8q7WPPSWJ0zwvdak61Qc4PXEXC72zkNS8KvHkPLZkQrvTsTe9jdU3PQpSnjxQOQm8rnDjPMElIr0UNOi730I0Ow4uKD0NtNU8OdsyvUMKVzys9Iu8rCpEvVVNyLsVnqS9EDx1vJu0PTyvdQ+9GsUeuzYTt7wuTmy9JQ+6PHM8V7xBHWm72P5tPtKAgD0S0p27bTLrvK0dzjxiRjm9ucH8O5h9PztpEBU9M/R/vScV4Lun6dk8/wnkvBwriDznbnm8KJ46PeQ7D71DIlI8o2ggPUEbKTzRzfY711jovEzUfDu6bIM7lukPvXNFm7tS7628VyoSva9lir0SG5G71+hEO7Z6jTwSeWM9TLjHvHQSVzyrzKI7SJJZPVH0m7zVQxy9pJgzPam0JruK4GK8GPT3vN6zJb24C5q8ppdZO1M7qzxBrrI8VfxNPNfucz0bjhg9VsXEvO6aFj37yAm97ElGvcFA6bzDv6u83R2vPJYtvztLxSq80D+pPGAIyjuXz/I8oP/JO1sLkT2jVDY8MPkOPY+6ibxMAi497W5GPegKhjsPsRC91tWGPBNGFTzmxms77fdnvIoV1TxlAbe8YIHzutZRcbuLx9s7mZ0Wveav2Txizna7rCkrPVJmTzylUiY9ewQIPcofQjyp7Tg8FeB1vJ79YztaALc8ifP3PC+qQDwH8py8oRpMu4lijDv9Qyq9VnFlPPIu8LwL6Zy8N6GCPHeEHb1C4HM9Qk0CvU/OqrwzpTg9ejEZvYfx/jx7a9a7dwoVPK+rE7wwAha9h54RvDLAlj1NKfw8ZFyvPHfpwzy/hVI8sUeWPJ9Emrs8GgW8g92TPQM99LzQsf68L04cOnRIWTz+kpW8GJcbPIU8Xzvjshi9fvWVvVaVFj1cTjQ9Ds42PMNuDT0rr/o7xG0EvVdYGDs8lC+9UmpgPRLDX7yVZHq8Q2NSvLlw5DwHPpM88SGivHeW7bzNfc67em8YPOUwUDzreKw8P7tAvfkoIDwTqJq8q4KQOx5lVDwJ0ig9Hd0KvD2K0TsU0Qq8AAavPLMjDj1dyi49AO8mvfxAhLxEL2I9Abfduxn1Aby45i09UdFuvdgOzrwDWCC9CeP+u+O/ZD1ZE8y8hGZ/PCFVf7zqNAA7TWjbO8LxjrshCZ86d9RKPIPmtrysQ3e8J6VNPZgrhTqdA5g9cEXpO78YT71bdaI9CAmgvChvB70l80W87s4zPdl4ZT2mRRk902FEPIzHAb1wU6q7VYVYPGBgLjtKCxs9r60LvWyNFb1C0Tw8tNWDO6E4m7yDXq48nDQsPCTvijy+xCm9pZWhPLae7DwlGUC9FEAIPOKyfj0zIEq8M0kEO1T3Fb2rwA28q5oxPIFsXL0amb69oORkvLXIPT3R6hi8ueAUvMxJBr0cTlK8XWa2vAhfljw8TbG7h9TWO+/VqTwednI8PbhVvGBPszulS3e7OVMfPb/OLzzzXFo9rEMhPSaKlzuGamA94XhpvYydlLzgv+k7diedPO2Vsr3UXl28wDtaOzrSFr2h+t28QWSAPZvRuTzW/KM8/8I8vZS7kbyiwNI7+VL5u8FdFj1gtZ+8PkQrPMHviLz7Nh09Cl3WuyLDOrxDdqi9kn9Uut7EqTzlH/M84MsyvYLdCTojOxe8AveyPPaLnjzXKTa8PQd/vKSHGL0zfkk8ZhtDPGELEDyaRGm75jFmvBqIXj3BYtu7N+V6PQtlrbyOTwu9BV21vKBFDb3YV8e8XCJVPevTJbwnxIw8Bl98PZxOLzupxSs8s96+u2ATlDlNEZ28f5m6vDbCkbyE3EO8KyyZu8s6Ab0z7Vw9xoxWPEzIEz3F9sS99+ouvIjNJz2CRgE9zxaEu+JpBjxo36g7vMlrvFKzwzww+jS82d9UPK2U3bxQc2E7bQklvRDYHD3/CMu74APBvN+cML3/lgw9lz/CuzBYL7xXORa8hoxvPDJKEL0nvLQ7sUuMPJWqybu9VQU9GrGHu2kdKTzDJUs7aQWpPC4T2DyJUOE8gs8BPM0GmTo9SV27PwRPvQ5ESLwfK6y8BV2sPPx0SztEvDW8jV6NPIeDD72S9E88JRSNvOj4NT1ImlG8SHa/vJSKuTxaf3S8GGnsu/AwhbyuFo888QzzvJzWJ72sR/y6xhwMvVJ+hzxLcwk92//yOcJQuzqL10+9gcnzOxOCDr1e5mi86uXrPEOmIbzbmwg8f1c/vUyvK7wYlUm8FDKhvMpVHT1oqmI89OubPI3tRz1jIbI8SMuNvIj/g7zGOUY8IrUavXsiAL71/3c6W/8/vee0h7zuulm9q5h8PB1r9LxWijG9llnRvJjhczyybyS9Yh0VPBMTtDxxov+8JhXzPEe5ED2bFQC9+StNvO4Lir3Cv249hZXmOv8Or7y0ilY8iCAdu7x/gj3ICTi9yMxOvQN5Ib2XUCS9xxl4veigQbxzSo+8H/iFPLCMJb3jXD28QqL7u48E2Dygb/W8URKmO9F76Dx14P+8OTsAPA5KmjyhehO8py6yvAiGYT0XqYC8qmFlvYT+CT0u6Ie89NyRPPYp8Dz+QX+7euIvPXBNDrxcFV89uKZKvGGGVb22LPy8a/wmPUcirLss3cm7L8rTuwH61Lzc4Nu8K1zLPNAV3Lxb6wI9JtGtvLEPMrw1qo289o+OvPWiAbwIfLu8r/DrvB2yUL3hu+a8pvxIvJoHvrw7peO7uBMZvULw2Dxu3oW9CBVZvYbHq7kAF0687GZNvdk/g7yqTsc75y06vAffT7zGzTe9ssoHvVEAXD3Mlpk8Jd1JPOnEO70z6dG8WCJUPKTx3rwc0bM716hZvWfIS7txJTU9ZteGvaBhoD3NtYm8wi3KO8jD1zsqTWi9RAP5PC2APjyYo9Y6uXjUvOH0vLz7j+Y7YzNuvNVUNT3KwW09Vp4YPXCF2TuTtqa8n7gou/JYDL0rpQu8x+jnvIIjgzm0JTY9sF+sPACSIL2Tj/w74c3FvFKpEz2XIeA82nx5PFT9Ab2rajU8GziVvK8FEb31nyQ8EzvIu71nab0EzWM8yOgevUCHgzwFyhm7ZRsGvOofDD0Seno8/vAyvJd6ZTs25/O8RihxPJFtPD34gL48yAqIPDEybzxtMDm8HOqZPPfdajw+RGa8Fy8lOrLug7xmfQC9iNE1vSgyFTxeOyC8zX1MvFkmO714JjI93oaLPCBPtLpEOhY7pyefPJP4DD2Wjka9yQ19u6YnKDyY3cU7NcQMPCdWdzrSbUQ9vJtvPJnTNTtjqTw95HsDPVpEjjzRUtk8vpPivInpmrybqhO9rjMlO/J2PTxjDY89OeKfPBJ87zzlUhy8c0eCPN1Cujw+bSA8nzQXPRWHMb3kdI88ikc7PIn/Ezx3fAW9ySTHPOuJ97z8fV+673p8PVEH+LzCBnu8Agq6vE91uD0Flpg7avTvvFXPhrtwLJw8zfczu9wd6jwmdB48XplTPZyGq7zW8Me82k/uvKoYsztJb9y8hrt9PHI/RLxkmrm8+9azvFmIM7x4uhe6SP+bPM06DbvjDze84E/dvBL55ju9Bdc7NP4DvajA1jx7tRO9Ikt3vWqr0Ly+Qbm8ZF/KPPtHILyP5Aq9q438u4DOFL2qUTw9Ehm4vLFmwzjtnAY8QznHuXsp1rtINhy95IlnvDQBdLtfxeI8HRprPDlFcLy3f6i86H2Du8ccBb2NMc28nr68uoAdAD1Riqi8aHhZPHE99TwYJeI7KryNu3zDpDs9ADS9X/FTvd7/Jj01H1C8gkkIu5NRqTyVxkC7iFhevCbK7rwVdhk8hORtPEgcnTyxt+C8djvAvOwaNzxyTO+8dhbTO9xsGL3qdBE8uRzmvF7Znbt/BRq8tDHpPDWDF72A1DO8eBOHPbBiqbwFSo28UD5GPJU0U71op1+8JB20PAu5K713vtM7Uwa5u19e3DycPdq6VLmauzcyxLzPPFW9n+ysPKRSXT0XS6O8N/mxu3Z3Qb0MEfo8UrBhPKwIGr2Sb7m9zzolu1oz1jxaSkI86DeMPQoHFD0hYgW9lXfDOykj2zsXKjk9If2YPKr/C72iyWK9+kAuPLcJOb2T6xO87W7dvGResLtGmq47EYNIvQhgQz3sChI96CGYPPbV0TxacLS7kPMVvZmBnbwXCMy8YjjAvOpXNj2I7RI9r4oVO6kdAjx0sWk8vT+oPB+S4bs2Llu5j7+2vMrjhTzz1u+87hcjvOS9rrzaU5Q84XKMPaXSWj0s1Se9fqmLOxCnz7sSmJY7GoV+u+zHsLyubHc9wc49PKqWojyWqKO8mNFKvZM3xbw9oDe8D1gSPGNwXjycvGs8GmTgOjZSiDwjKgu93+sRvMgEOL2o5e48yat3vUaQHz1LtQI6tj+JPIdmq7wS5xs98mgOPZEbNj1QWVi8FvvwPHctdr3xUpa8djlPvL9YNj1ySJ68pVtMPIvFAT0s0FU8VfhTvXTgbDtC7ly9MhkBPYxCaz2gNp+7r8vavJYV7zzRMMW8/uKwPM7e0bxOUbe6omgiPTELFTwEZJQ8bMq0PJFcQL2MrRO9UCkPPfK5S72Qzpc9m6FcPML6qzwEytY8NAp3PL/DvjvjJWq6r3JKvPCjq7w1mO680qdUPGbJJT01yQK9xSvlu7CFOzyZB7+7RmmZvCxlD738gw89rEYJvPDNUb0YU1O8IdppOzm3eb1Of8w8ZXwCPQ4h3L22Bbs8OkqEu1DMpbvQY6I8F70MvYjs1DwLgzq8WUmavUF8Az0+gCo7rFj9PNg9rjx/fR09xleAPHZDBD3oRsy8EXUfvMGuNLyKdew8JyYAvGO7CbzhKbi8wgXHvDXZlLwjtMk9zlJDvKmcS73aR8A8beiDOl63sLy4h508iyugvKL5Cz13sX68ly9ePG1B0zqoMhG9dIWsO7o4Szun/B28C4YJvEoklzqRE848t27KPHbkID0sKRO84zJEvbNSWTug0F48bVTau2yIoTtzkTM9AWEUvaIrAj2V7/k8jm4nveZdQrsyWAC9tpYkPTiMS7v2wfk7BX+fvZj7Ez2Sh768LpYpPSVrhLzm5pA8xPOvuuBHTz3SIpA8UTlQu2SGcz3ejS091siSvEmJTT31Riq9HMnevGZ8qrw2EbY8LxPcPM/45jsNTSk9LHhoPTTRLLwjrAG9rIWNPJqlvbwjJSq+roliPCuE4DvfSBm9sBNWvEWJCz3r+Ne699OzvG4lIbwijkg8QgWDPK5qEb36Xuo8a4E/OSbiHT3HeOC8be2bu0wvdbxf2w6875WWuwAlNr3YJ6W8rfILPbLznLyGiao8o6kiPbxJQT01xP86FToKvYLUjLtpCrQ7j+d3vMg9PrzYaRq9c4EUPDLNDj3rWys94cg5vT1XKz1oODI9bTq1PLgHSz0AxBy8fsE7Pda20TxVtLi8nmQuvV1Serz/Cla8NmVEu8NJJ70Japa806TvvKI+XLw7kA69ddI2PN0+nDxng7Q8MW59PSQhST0GWla8dwZ2vZnbrDxcWBe94yAhvZpQUTxaeQY8cFusPEbECzzwYje8WcG0vC5LHL0ObgI9YXiAPOf+NbujGwA85LmCPMQi37zXESi9kMOkvZ2sW7wJiwy92LeEvKwCwjy59BC9UXPNvT2cJT2nHIy8qQK0PMwXij6aS2I9fuV2vMqXGr2VX7a8oB8Bva/AubyNZAu8yfJwPAwHLL0n6P+8e/UovL3NOjwYnc88/oPYvGYseT2Q7C+904MtuqF1Nz1gfCc8TJpKOsG8+rwI4F88TKbEu8E2ubylXZy7xYomvSu/5bxk6Vm9EjuJOzV6kLumbrY79hMmPQAGOb2dgjU8YQO5vBrtNT02HJg8bUT+vPjihzzqZ3q7hGU0vG3oDL2Dro27MaoVvFlQGDwNV0c78z3BPOe6nzxmApY8LZsCPdmYXL21+Xu8x+OFvF+UVb3pmfG8nFSpvNZWFrzZFZ28rfMKPblk+zxG8Yk8NegUvKx8cbypXlw9z/juuiz1mTylLh68baxvPaGhpjw4zpc73/PNvNJB3Lz4JjY8oQfGO3O55Lys2Ak9E8PrOULCXziN9208WYWBvNPitryTqT+8aNsCvST1aDx00Ai6Y+0IPP62iD2jsjS9PFtsPLXtkLytGV27qm9dvB84ND3Dxtm8wR2UuplrjTyQABM99v52vaCJ7LvkAe67nI44vHMBT7ye0Wq8w+Y+PWa91bwTQiC8Ozj/PJbfKr1Z6wE9O+XNvCZgE7ttY6O7F5l2va1RUjzU5hA9vALhPEIauDpoyW095SaEvKjXSTtrjPU7gnBAvcjxwLzje5a82z0EvFXBxrwxaDM8gxK+OycHl7y2rqS72mwkvYktpb3T7m47kYkePeqyKrxzr/Q8M8PIOzW05rxFxVm7cvRBvRYyyTxej0G82Fy7POHiRbyiOzk8TMdFvX4U8DtsJEG8L6KovCFGV7w6u4M8DU/mPP+l+rzeyfg8/3QQvWHOZzwxiE+8dC6RPLRinrypq4U85ucxvcsTCz0RCZM957RPPXUVpzzRIA28VaW9PElmEb0eJN08OG/3PHRNmrwV4iW8nVudvHgU/jzZJ7k8w+DuvCJYjDzA6JS8U0bDPNAw7bvWuxE92Oa7PLZ7Hz2YyXA8d0OOvI1TVT1XkNa8Rm5xPQkMyrxbf0m9LpaIPQJgC7u9KLW8PdLvvMbLXz2lyIw8rF3KPKlNFDwftdc5QZ+ovPmBQr1DH8Y8ZUqEPTRI2rywHUC9O+NbuylNeboJGto7EZ9cPbvTFDzmEgQ94ZWyu8818TyHEF88JxB4vA7sDj1V0tU8Okm3PAk/1LwP0K68qkFYPJVFKT3994e9VaYjvcMSkjwqknM8qLxYuunFkDyBvgy8tsKhu/prRL2HGSY8rS1UvLXnj7wu9PG7YCRCO8sqnTvEyNO77tqGu0780zw26DI8394OPUiRKj0WuzU8SnmePTx1e71jkK48pLbFO3NNMT0x4ai9uDGhvPIvvLvuw9+8LPNdPP6oHD2UOf28Wpm3PDNdRb1dUDA6lElcvIfsDDzYCeI8dxYcvH/5L7w4q/a8FOgzPThrh7wZXg68RcBgvTa7uDz9BI88Gvx9uwLwgrxQZIS85xYKPPNybzstiD09TTcsPW3jDLQmqZm8QfoIvPMpJDxvt7c8foVjO+eGajylAAw8AWHWvJfBiT0nXxg9p08CvWBdCLsc9Sy9uWncPKT75jwhbAM9g54DPIHNJT3lRgW9n3m3uwcg5LyBI5m6ZmREvR2PWL12c468mmQnOypSULujxoq78e1PO4Z6MjzpgbE8vCK4vdwuwrscBS68B/RKPR/QULwwLtM8VW6yO+hh1TsSEqa6yM5Pu+bbRz1GRZm8Qjumu5Ckcr0NPmM9zGxRvWeezbueowq9mRoTPcymk70vkzY83mkYvVz2L70KJg+8QEGUPIaPqTzW7w29UDDpPGhXgbyMhs+8CXs7vYXBkTz+WCg89A4tPOjP8Tt59Zo80EkxvP9LKb0Z3g67NhX0vNOcTbutPw+867S0uy0mWbxBdzy7vrozPS218DsyhAE9gh+NvDZMsLycYHg9dUftu/M0OTw5iIg8KNDDO6K9Cb1mejW9s7UxPITl6bxFKvU8/1GIPAgB4Tx/aGk89qYgvaRkizxFkxm9HqS8vJX1vDvxKog890/QvICReb3oMQc9xJESPMaxgbvzyyM8GXxDvB8xlzwbG+K8xnjUPPzw4bu3X6q8j10KujWFDL27Lw2+NV69vPWRxrwtKu46eB9svXaN6Dz1AVe9qnNdu779V7zr4lE8Nyd3vSjoEzv6ME89wdTIvIrFezyQiEA9nHfZuwTJI7xAVjy9N+F9PcmLO7vGG7a7RJjRPGCIhzxPycM8UU0RvawfGr0Q0BW9CzYFvUYRjLxSCOo8jNvDOwg9UTy2wGc7+dH1vGrIDb3uIzw92IMNvR5HsrupUM08rizwvE+9uTveszA9Yai7vDN2Br2DcV89+BXIvDNe27w6Y/s72nNBvKK9krt4UGM8/YE8PD5Dgz1Hauo7ol0zPQeafLyAT9G8zGWjOSRdaj2mDNO8SVhuPGcwqTvYwW68qKTpvAigTj3zTeq8tOyUPN4dqLpNDAq8zJnWvLN+Ob2F5wi9NflBvCqClLyQ/7U76AvAPOjgDD0cGKa8IvurvNjLFL0NdCK8ZuP6vB4ACb2Ydek7gR5uvO6SDbwH0qi8R1fLvOwii7vFyB28OuEnvT7Nkrz8pB89OsIlPdj9DTytTCO91o7ivLeaIzzBxtG8qbRTPPowjL3fZS08Ti9LPebLbb1m83k9bumoumg0Brs7Gb+8PCG3vPKwTj2eOxo9Jv17uzFHe7xlQhq7WqydPL7Mcj0c9ik9MmOnPAe6+Tymook8LZXOvNmFBr27rAy9W7/ivM/QLryfsXE7VdANPQmXmDw6+uC8m0+sNoBUyrw8+u+7hwYDPQlv0zzuoOW8wDulvJZaR7wI1Xa8h6OgvI7eHruDN2e8uFbauoHJv7yrgbM8Ol0HPYFex7uR8mI71zQuPVwAjbtUw9M8yM7bvCGnJz26kkA91NNCPVC/qzxfGK68TqxsPNTudDzSCqa7pxoDPHmVAr3VbN+8Oq1jvZM21byWzcQ8Io6tuhYfo7u+pJq84Fe7PMmRjzyF/HY8GGKMvKostDxxo5M8UnoRvE3YPLy3rIk8U3b9PLcbnLz+wdg8l8QSvFIFbjxcOmc8tUTuPP4IADzq3wE80sOTPNA+47v+Q0465BA9vaC/vLsROkc8jFlePbLavbkmyuk81RmYvFNZuTzq5Os7QCynPCd+6buJEfi8XCoDPMstobvzcMU8a6KPvL0wJT1dXOW8ISiLvEemAj3LNQe9JSmWO2QyEjy0SWs9MAckvMDLUb2fI2w8MsE6PWPLRTwABB89U4hkPDrZwzw6kpM8NfhFvRWz1jvhoPq4UCiYPItFujwHmz48++yxO7Nnq7xTnr67wrZuu6dM7DskK8k8j3+pu7cd/LyT1BO6HzA9vNAgSTyme3g9OWm+vBR8mr2SnwW9'
def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def text_sha(value):return hashlib.sha256(value.encode('utf-8')).hexdigest()
def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix('.partial.json');temp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8');temp.replace(path)
snapshot=Path(snapshot_download(MODEL,revision=REVISION,allow_patterns=list(EXPECTED_FILES),cache_dir='/tmp/lexia_hf'))
for name,digest in EXPECTED_FILES.items():assert sha_file(snapshot/name)==digest, 'Arquivo do modelo divergente: '+name
model=SentenceTransformer(str(snapshot),device=DEVICE,local_files_only=True)
model.float();model.max_seq_length=8192
assert model.get_embedding_dimension()==1024
assert all(p.dtype==torch.float32 for p in model.parameters() if p.is_floating_point())
OUTPUT=Path('/kaggle/working/lexia_embeddings');OUTPUT.mkdir(parents=True,exist_ok=True)
print('Backbone verificado; dimensão 1024; precisão float32.')

## Compatibilidade numérica com legislação

Três pequenos textos normativos e seus vetores originais são usados somente como prova técnica. O teste confronta os IDs de tokens e os embeddings da GPU, com tolerância explícita; não regrava nenhum artefato de legislação.

In [ ]:
TOLERANCIA_FP16=0.9995
def encode_vectors(texts,batch_size=32,fp16=True):
    """FP16 por autocast; saída renormalizada e gravada em float32."""
    with torch.autocast('cuda',dtype=torch.float16,enabled=fp16):
        v=model.encode(texts,batch_size=batch_size,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
    v=np.asarray(v,dtype=np.float32)
    return v/np.linalg.norm(v,axis=1,keepdims=True)
for p in PROBES:
    assert text_sha(p['texto'])==p['sha256']
    assert model.tokenizer.encode(p['texto'],add_special_tokens=False,truncation=False,verbose=False)==p['tokens'], 'Tokenização incompatível.'
reference=np.frombuffer(base64.b64decode(PROBE_VECTORS_B64),dtype='<f4').reshape(3,1024)
actual=encode_vectors([p['texto'] for p in PROBES],fp16=False)
cosines=np.sum(reference*actual,axis=1);max_error=float(np.max(np.abs(reference-actual)))
assert np.isfinite(actual).all() and float(cosines.min())>=.99999 and max_error<=1e-4, 'Compatibilidade numérica não confirmada.'
actual16=encode_vectors([p['texto'] for p in PROBES],fp16=True)
cosines16=np.sum(reference*actual16,axis=1)
assert np.isfinite(actual16).all() and float(cosines16.min())>=TOLERANCIA_FP16, 'Compatibilidade em FP16 não confirmada.'
PROFILE=dict(modelo=MODEL,revisao=REVISION,arquivos_sha256=EXPECTED_FILES,dimensao=1024,dtype='float32',normalizacao_l2=True,
    precisao_calculo='fp16_autocast',armazenamento='float32_renormalizado',tolerancia_fp16=TOLERANCIA_FP16,
    device=DEVICE,gpu=torch.cuda.get_device_name(0),versoes=VERSIONS,cosine_min=float(cosines.min()),erro_max=max_error,
    cosine_min_fp16=float(cosines16.min()),erro_max_fp16=float(np.max(np.abs(reference-actual16))))
atomic_json(OUTPUT/'compatibilidade_gpu.json',PROFILE)
print('Compatibilidade float32:',float(cosines.min()),'erro máximo:',max_error)
print('Compatibilidade FP16:',float(cosines16.min()),'erro máximo:',PROFILE['erro_max_fp16'])

## Benchmark técnico e dimensionamento

O teste usa oito textos artificiais de 500 tokens e mede custo, não qualidade jurídica. Use a velocidade observada para planejar os lotes dentro da quota exibida pela sua conta. O lote inicial é pequeno; mantenha esta configuração antes de explorar outras precisões ou múltiplas GPUs.

In [ ]:
synthetic='Texto artificial para medição de tempo. Não representa decisão judicial. '*500
offsets=model.tokenizer(synthetic,add_special_tokens=False,return_offsets_mapping=True,verbose=False)['offset_mapping']
samples=[synthetic[:offsets[499][1]] for _ in range(8)]
torch.cuda.synchronize();started=time.monotonic()
test=model.encode(samples,batch_size=8,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
torch.cuda.synchronize();elapsed=time.monotonic()-started
BENCHMARK=dict(entradas=8,tokens_por_entrada=500,segundos=elapsed,chunks_por_segundo=8/elapsed,
    horas_por_milhao=elapsed/8*1_000_000/3600,amostra='artificial; não é previsão garantida nem métrica jurídica')
atomic_json(OUTPUT/'benchmark_gpu.json',BENCHMARK)
print(BENCHMARK)

## Produção por pacotes — somente após validar o notebook 05

Os pacotes de jurisprudência devem ser adicionados como **Kaggle Dataset privado**. Não envie o repositório inteiro, `.env`, credenciais ou originais anteriores ao mascaramento. Cada pacote contém apenas `chunk_id`, `registro_id`, `texto` e `texto_sha256`, além do manifesto. O texto pode conter nomes; privado não significa anonimização integral.

Informe em `JOB_IDS` os pacotes desta sessão, em ordem, e habilite `EXECUTAR_PACOTE`. Eles são processados em sequência até `MAX_SECONDS` (11 h, com margem para salvar). Cada parte concluída vira um checkpoint. Se a sessão acabar, salve os outputs e anexe-os como Input da próxima sessão em `PREVIOUS_OUTPUT`: as partes já concluídas são conferidas e reaproveitadas.

Em cada parte, uma amostra de textos reais é recalculada em float32. Se o cosseno mínimo ficar abaixo de 0,9995, a execução para sem gravar a parte.

In [ ]:
EXECUTAR_PACOTE=False
JOB_IDS=['job_0000']          # pacotes desta sessão, processados em sequência
BATCH_SIZE=32
MAX_SECONDS=11*3600           # margem para salvar antes do limite da sessão
AMOSTRA_FP32=8                # textos por parte recalculados em float32
PREVIOUS_OUTPUT=None          # Ex.: Path('/kaggle/input/meu-output-anterior/lexia_embeddings')
def safe_child(root,relative):
    root=Path(root).resolve();target=(root/relative).resolve()
    assert target.is_relative_to(root), 'Caminho fora do pacote.'
    return target
def verify_output(input_rows,folder,cp):
    vf=safe_child(folder,cp['vetores_arquivo']);mf=safe_child(folder,cp['mapeamento_arquivo'])
    assert sha_file(vf)==cp['vetores_sha256'] and sha_file(mf)==cp['mapeamento_sha256']
    vv=np.load(vf,allow_pickle=False);mm=pq.read_table(mf).to_pylist()
    assert vv.shape==(len(input_rows),1024) and vv.dtype==np.float32 and np.isfinite(vv).all()
    assert np.allclose(np.linalg.norm(vv,axis=1),1,atol=1e-4) and len(mm)==len(input_rows)
    for i,(r,m) in enumerate(zip(input_rows,mm,strict=True)):
        assert m['row_index']==i and m['chunk_id']==r['chunk_id'] and m['registro_id']==r['registro_id']
        assert m['texto_sha256']==r['texto_sha256'] and m['vector_sha256']==hashlib.sha256(np.asarray(vv[i],dtype='<f4').tobytes()).hexdigest()
if EXECUTAR_PACOTE:
    import shutil
    started=time.monotonic();sem_tempo=False;sessao=[]
    manifestos={}
    for path in Path('/kaggle/input').rglob('manifesto_job.json'):
        candidate=json.loads(path.read_text(encoding='utf-8'));manifestos.setdefault(candidate.get('job_id'),[]).append((path,candidate))
    for JOB_ID in JOB_IDS:
        found=manifestos.get(JOB_ID,[])
        assert len(found)==1, 'Anexe exatamente um pacote correspondente a '+JOB_ID+'.'
        manifest_path,job=found[0];jobroot=manifest_path.parent
        assert job['corpus_validado'] is True and job['modelo_arquivos_sha256']==EXPECTED_FILES
        assert job['chunks_total']<=100000, 'Pacote maior que o limite desta configuração.'
        jobhash=sha_file(manifest_path);dest=OUTPUT/JOB_ID;dest.mkdir(parents=True,exist_ok=True);completed=[]
        for part in job['partes']:
            inp=safe_child(jobroot,part['arquivo']);assert sha_file(inp)==part['sha256']
            rows=pq.read_table(inp).to_pylist();assert len(rows)==part['chunks']
            for r in rows:assert text_sha(r['texto'])==r['texto_sha256']
            label=part['parte_id'];cpfile=dest/('checkpoint_'+label+'.json')
            prior=Path(PREVIOUS_OUTPUT)/JOB_ID if PREVIOUS_OUTPUT else None
            if not cpfile.exists() and prior and (prior/cpfile.name).exists():
                cp=json.loads((prior/cpfile.name).read_text(encoding='utf-8'))
                assert cp['job_sha256']==jobhash and cp['entrada_sha256']==part['sha256']
                verify_output(rows,prior,cp)
                for name in (cp['vetores_arquivo'],cp['mapeamento_arquivo'],cpfile.name):shutil.copy2(safe_child(prior,name),dest/name)
            if cpfile.exists():
                cp=json.loads(cpfile.read_text(encoding='utf-8'))
                assert cp['job_sha256']==jobhash and cp['entrada_sha256']==part['sha256']
                verify_output(rows,dest,cp);completed.append(cp);continue
            if time.monotonic()-started>=MAX_SECONDS:sem_tempo=True;break
            unique={r['texto_sha256']:r['texto'] for r in rows};keys=list(unique);encoded={}
            for a in range(0,len(keys),BATCH_SIZE):
                if time.monotonic()-started>=MAX_SECONDS:sem_tempo=True;break
                kk=keys[a:a+BATCH_SIZE];texts=[unique[k] for k in kk]
                assert all(len(model.tokenizer.encode(t,add_special_tokens=True,truncation=False,verbose=False))<=502 for t in texts)
                vv=encode_vectors(texts,batch_size=BATCH_SIZE,fp16=True)
                for k,v in zip(kk,vv,strict=True):encoded[k]=v
            if sem_tempo:break # Apenas a parte aberta é refeita; checkpoints anteriores continuam válidos.
            amostra=keys[:AMOSTRA_FP32]
            ref32=encode_vectors([unique[k] for k in amostra],batch_size=BATCH_SIZE,fp16=False)
            cos_amostra=float(np.min(np.sum(ref32*np.stack([encoded[k] for k in amostra]),axis=1)))
            assert cos_amostra>=TOLERANCIA_FP16, f'Parte {label}: amostra FP16 diverge do float32 (cosseno {cos_amostra:.6f}).'
            vv=np.stack([encoded[r['texto_sha256']] for r in rows]);mapping=[]
            assert vv.shape==(len(rows),1024) and vv.dtype==np.float32 and np.isfinite(vv).all() and np.allclose(np.linalg.norm(vv,axis=1),1,atol=1e-4)
            for i,r in enumerate(rows):mapping.append(dict(row_index=i,chunk_id=r['chunk_id'],registro_id=r['registro_id'],texto_sha256=r['texto_sha256'],vector_sha256=hashlib.sha256(np.asarray(vv[i],dtype='<f4').tobytes()).hexdigest()))
            vf=dest/('vetores_'+label+'.npy');mf=dest/('mapeamento_'+label+'.parquet')
            vt=vf.with_suffix('.partial.npy');np.save(vt,vv,allow_pickle=False);vt.replace(vf)
            mt=mf.with_suffix('.partial.parquet');pq.write_table(pa.Table.from_pylist(mapping),mt,compression='zstd');mt.replace(mf)
            cp=dict(job_sha256=jobhash,entrada_sha256=part['sha256'],parte_id=label,chunks=len(rows),
                vetores_arquivo=vf.name,vetores_sha256=sha_file(vf),mapeamento_arquivo=mf.name,mapeamento_sha256=sha_file(mf),
                precisao_calculo='fp16_autocast',armazenamento='float32_renormalizado',amostra_fp32=len(amostra),cosseno_min_amostra_fp32=cos_amostra)
            verify_output(rows,dest,cp);atomic_json(cpfile,cp);completed.append(cp)
            print(JOB_ID,'| parte concluída:',label,len(rows),'chunks | cosseno amostra float32:',round(cos_amostra,6),flush=True)
        total=sum(c['chunks'] for c in completed)
        status='job_concluido' if total==job['chunks_total'] else 'job_parcial'
        atomic_json(dest/'resultado_job.json',dict(status=status,job_id=JOB_ID,job_sha256=jobhash,chunks_concluidos=total,
            chunks_esperados=job['chunks_total'],perfil=PROFILE,partes=completed))
        sessao.append(dict(job_id=JOB_ID,status=status,chunks_concluidos=total,chunks_esperados=job['chunks_total']))
        print(JOB_ID,'|',status,'|',total,'/',job['chunks_total'],'chunks',flush=True)
        if sem_tempo:break
    atomic_json(OUTPUT/'resumo_sessao.json',dict(jobs=sessao,segundos=time.monotonic()-started,limite_atingido=sem_tempo))
    print('Sessão:',sessao,'| tempo esgotado:' if sem_tempo else '| concluída em',round((time.monotonic()-started)/3600,2),'h')
else:
    print('Somente configuração e benchmark. Nenhum corpus de jurisprudência foi vetorizado.')

## Salvar a versão e os arquivos

Use **Save Version → Save & Run All**, mantendo o notebook privado, e confira os arquivos em Output. Preserve `compatibilidade_gpu.json`, `benchmark_gpu.json` e a pasta do job. Baixe os resultados ou anexe o output salvo como Input para uma retomada; arquivos temporários do modelo não entram nos resultados.

O limite deste notebook é 100 mil chunks por pacote: aproximadamente 410 MB de vetores float32, mais mapeamentos. A base inteira ultrapassa a capacidade de saída de uma única execução; processe e salve os pacotes separadamente. A importação final no pgvector será feita no computador após conferir hashes, contagens e compatibilidade de todos os resultados.